![DB Academy](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/db-academy.png)

# Demo - Designing Datasets for Dashboards

## Overview

Databricks AI/BI Dashboards help you quickly transform data into shareable insights. **Datasets** are the sources of information for AI/BI Dashboards and the foundation of every visualization you build.

In this demo, you will build the datasets for a **Canada Sales** dashboard. You'll start simple with a **pass-through** dataset from `ca_opportunities`, then add a **multi-table join** across `ca_orders`, `ca_customers`, and `ca_products`, and finish with an **aggregation** dataset that combines `ca_sales_targets` with `ca_orders` so the dashboard can compare yearly targets against actual revenue.

By the end of the demo, your dashboard will have **three well-named datasets** ready for visualizations in the next demo.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Create and name a new AI/BI Dashboard**.
2. **Select a SQL warehouse** for the dashboard to query against.
3. **Add a pass-through SQL dataset** from a single Unity Catalog table (Sales Opportunities).
4. **Add a multi-table join SQL dataset** (Full Orders Data).
5. **Add an aggregation SQL dataset** with HIT / MISS variance logic (Sales Targets).
6. **Name and organize datasets** so the dashboard is easy to reuse later.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Before starting this notebook, make sure you ran the **01 Demo - REQUIRED - Data Setup and Exploration** notebook once to set up the course data.
  </div>
</div>

## A. Classroom Setup

1. Run the cell below to:

   - Confirm tables were created in the **01 Demo - REQUIRED - Data Setup and Exploration** notebook

   - Set your default catalog and schema

In [0]:
%run ./Includes/Classroom-Setup-ca_setup

Workspace,User Catalog Name,Catalog Status
Vocareum Workspace,labuser17404915_1791275222,Catalog Already Exists


Status
✅ All required tables found in labuser17404915_1791275222.aibi_dashboards (5 tables).


Course Catalog,Your Schema
labuser17404915_1791275222,aibi_dashboards


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Please Note
  </strong>
  <div style="color:#333;">

All tables referenced throughout this demo live in <strong>labuser_UNIQUE_ID.aibi_dashboards</strong>.

Many of the following steps happen in a <strong>non-notebook area</strong> of the Databricks platform. It's recommended to open the dashboard editor in a <strong>separate tab or window</strong> so you can reference these instructions alongside it.

  </div>
</div>

## B. Create and Name Your Dashboard

1. In the main navigation bar, right-click **Dashboards** and select **Open Link in New Tab**.

2. Click **Create dashboard** in the top right.

    At the top left of the dashboard definition panel, you'll see a placeholder name like `New Dashboard 202X-01-01 12:00:00`.

3. Click on the placeholder name and change it to `FirstName-LastInitials - Canadian Sales`.

## C. Select the Dashboard Compute
1. At the top right select your **Serverless SQL Warehouse** (in the provided lab it should be called **shared_warehouse**).

#### Checkpoint
![Warehouse](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/designing-datasets-for-dashboards/select-warehouse.png)

## D. Explore the Dashboard

1. In the dashboard, you can:
   - Select the **compute** to use
   - **Publish** the dashboard
   - **Share** the dashboard

2. Select the empty page (**Untitled page**)
     - The main page **Canvas** includes a toolbar to:
       - Create a dashboard with Genie Code
       - Create manually
         - Add visualizations
         - Add a text box
         - Add filters
         - Undo and redo actions

3. Select the **Data** tab in the top left of the canvas to add data.

   You will see two options:

   | Option | Description |
   |--------|-------------|
   | **Add dataset** (metric views are outside the scope of this course) | - **OPTION 1** - You can reference a **Unity Catalog registered metric view** <br> - **OPTION 2** - You can reference a table and it will create a **dashboard-local metric view** where you can define dimensions, measures, and joins using a visual interface. *Generally available.* <br/>Dashboard local metric views documentation: [AWS](https://docs.databricks.com/aws/en/dashboards/manage/data-modeling/local-metric-views) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/manage/data-modeling/local-metric-views) \| [GCP](https://docs.databricks.com/gcp/en/dashboards/manage/data-modeling/local-metric-views) |
   | **Add SQL dataset** | Lets you write a SQL query to define your dataset for the dashboard (can query a table or metric view). |

   **Create and manage dashboard datasets** - [AWS](https://docs.databricks.com/aws/en/dashboards/manage/data-modeling/datasets) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/manage/data-modeling/datasets) | [GCP](https://docs.databricks.com/gcp/en/dashboards/manage/data-modeling/datasets)


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    NEW - Dashboard Relationships
  </strong>
  <div style="color:#333;">

- Dashboard relationships let you model the join logic between your datasets in AI/BI dashboards, so you can build multi-fact, multi-grain data models and reusable cross-dataset measures without pre-joining data in SQL. You model the relationship once, then use it across every visualization in the dashboard. These are outside the scope of this course, but take a look at the documentation.

[AWS](https://docs.databricks.com/aws/en/dashboards/manage/data-modeling/dashboard-relationships/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/manage/data-modeling/dashboard-relationships/) |
[GCP](https://docs.databricks.com/gcp/en/dashboards/manage/data-modeling/dashboard-relationships/)



  </div>
</div>


## E. Add Data to Your Dashboard

Let's build queries to use as our dashboard's source data. 

You'll create **three datasets**, each answering a different kind of question:

| Dataset | Purpose |
|---|---|
| **Opportunities** | Open, won, and lost deals. |
| **FullOrdersData** | Row-level orders enriched with customer and product details. |
| **SalesTargets** | One row per year comparing target vs. actual revenue, with HIT / MISS flag. |




<div class="scn-wrap">

<style>
.scn-wrap {
    width: 100%;
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 16px;
}

.scn-header {
    text-align: center;
    max-width: 1100px;
}

.scn-header-title {
    font-size: 20pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 6px;
}

.scn-header-sub {
    font-size: 14pt;
    color: #5E7077;
    line-height: 1.5;
}

.scn-row {
    display: flex;
    justify-content: center;
    gap: 20px;
    flex-wrap: wrap;
    max-width: 1200px;
    width: 100%;
}

.scn-card {
    flex: 1 1 320px;
    max-width: 360px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 22px;
    position: relative;
    box-sizing: border-box;
    color: #0b2026;
    display: flex;
    flex-direction: column;
}

.scn-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.scn-card.s1::before { background: #4299E0; }
.scn-card.s2::before { background: #00A972; }
.scn-card.s3::before { background: #FF5F46; }

.scn-badge-row {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-top: 8px;
    margin-bottom: 12px;
    flex-wrap: wrap;
}

.scn-option-num {
    font-weight: 700;
    font-size: 14pt;
    letter-spacing: 1.5px;
    text-transform: uppercase;
}

.scn-card.s1 .scn-option-num { color: #4299E0; }
.scn-card.s2 .scn-option-num { color: #00A972; }
.scn-card.s3 .scn-option-num { color: #FF5F46; }

.scn-you-here {
    display: inline-block;
    background: rgba(255,95,70,0.15);
    color: #FF5F46;
    font-weight: 700;
    font-size: 12pt;
    border-radius: 4px;
    padding: 2px 10px;
    letter-spacing: 0.5px;
}

.scn-title {
    font-size: 16pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 12px;
    line-height: 1.3;
}

.scn-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.5;
    margin-bottom: 14px;
    flex: 1;
}

.scn-text ul {
    margin: 0;
    padding-left: 20px;
}

.scn-text li {
    margin-bottom: 10px;
}

.scn-text li:last-child {
    margin-bottom: 0;
}

.scn-code {
    font-family: Menlo, Consolas, monospace;
    font-size: 11pt;
    background: #EEEDE9;
    border-radius: 4px;
    padding: 10px 12px;
    color: #0b2026;
    margin-top: auto;
    white-space: pre-wrap;
    word-break: break-word;
    line-height: 1.3;
}

.scn-footer {
    max-width: 1100px;
    text-align: center;
    font-size: 14pt;
    color: #5E7077;
    font-style: italic;
    line-height: 1.5;
}
</style>

<div class="scn-header">
  <div class="scn-header-title">Your situation may look different when adding data sources to your dashboard</div>
  <div class="scn-header-sub">
    Pick the path that fits your access. Your options depend on what your data team has built and your permissions.
  </div>
</div>

<div class="scn-row">

  <div class="scn-card s1">
    <div class="scn-badge-row">
      <span class="scn-option-num">Option 1</span>
    </div>
    <div class="scn-title">Final Joined Table or Metric View Already Exists</div>
    <div class="scn-text">
      <ul>
        <li>Denormalized <strong>joined tables, views, or metric view</strong> already exist in Unity Catalog.</li>
        <li>You have <strong>SELECT</strong> access to the object.</li>
        <li>Add final object<strong> directly in your dashboard</strong>. No additional join SQL needed.</li>
      </ul>
    </div>
    <div class="scn-code">SELECT * FROM object</div>
  </div>

  <div class="scn-card s3">
    <div class="scn-badge-row">
      <span class="scn-option-num">Option 2</span>
      <span class="scn-you-here">this demo</span>
    </div>
    <div class="scn-title">You Have SELECT on Source Tables Only</div>
    <div class="scn-text">
      <ul>
        <li><strong>Read-only</strong> access on the <strong>normalized</strong> tables.</li>
        <li>Cannot create objects in Unity Catalog.</li>
        <li>You can create a <strong> local metric view</strong> or <strong>write the SQL</strong> logic as a SQL dataset in the dashboard editor.</li>
      </ul>
    </div>
    <div class="scn-code">SELECT ... FROM ca_orders o INNER JOIN...</div>
  </div>

  <div class="scn-card s2">
    <div class="scn-badge-row">
      <span class="scn-option-num">Option 3</span>
    </div>
    <div class="scn-title">Full Pipeline and Object Creation</div>
    <div class="scn-text">
      <ul>
        <li>Ability to <strong>create tables, views, or metric views</strong> in Unity Catalog.</li>
        <li>Ability to build objects and pipelines with <strong>Apache Spark Declarative Pipelines</strong>, <strong>Lakeflow Jobs</strong>, <strong>SQL views</strong>, or <strong>metric-view YAML</strong>.</li>
        <li><strong>Reuse</strong> created objects across many dashboards.</li>
      </ul>
    </div>
  </div>

</div>

<div class="scn-footer">
  Same result in all three cases. The difference is where the SQL logic live: In the dashboard, a metric view, or a pipeline.
</div>

</div>

### E1. Add a Dataset for Sales Opportunities

1. Still in the **Data** tab, select **Add SQL dataset**.

2. Select your default catalog and schema:
    - **Catalog**: **labuser_UNIQUE_ID**
    - **Schema**: **aibi_dashboards**

3. Enter the following query into the dashboard query editor:
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <div class="code-block" data-language="sql">
SELECT *
FROM ca_opportunities
  </div>
</div>

4. Click **Run** to execute the query and examine the results.

5. Rename the **Dataset** to `Opportunities`.
    - **NOTE:** Click the **kebab menu** next to the dataset and choose **Rename**.



<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size: 14pt;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:16x;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

#### Checkpoint

| opportunityid | customerid | productid | province | date       | amount | salesrep | phase           |
|---------------|------------|-----------|----------|------------|--------|----------|-----------------|
| CA428145      | TC0514     | JQ9322    | ON       | 2024-08-22 | 344    | Sam      | 3-Closed Lost   |
| CA428146      | NF9938     | W8931     | QC       | 2024-09-11 | 162    | Ethan    | 0-Prospecting   |
| CA428147      | NR9560     | BC111d    | QC       | 2024-09-27 | 90     | Ethan    | 1-Active        |
| ...           | ...        | ...       | ...      | ...        | ...    | ...      | ...             |

#### E1.1. Add a Custom Calculation for `deal_size`

A **custom calculation** adds a new column to the dataset, computed from existing columns. It lives alongside the dataset's output and is available to every widget that uses this dataset. This lets you add dashboard-specific logic without editing the source SQL.

1. In the result window, select **Add custom calculation**.

2. Configure the following:
   - **Name**: `deal_size`
   - **Comment**: `Buckets the deal opportunity into Small, Medium, and Large`
   - **Expression**:
     ```sql
     CASE
       WHEN amount < 500  THEN 'Small'
       WHEN amount < 5000 THEN 'Medium'
       ELSE 'Large'
     END
     ```

3. After saving select the **Schema** tab in the result window.
    - Confirm a new `deal_size` column appears at the end.

**What are custom calculations?** - [AWS](https://docs.databricks.com/aws/en/dashboards/manage/data-modeling/custom-calculations/) | 
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dashboards/manage/data-modeling/custom-calculations/) | [GCP](https://docs.databricks.com/gcp/en/dashboards/manage/data-modeling/custom-calculations/)

### E2. Add the Full Orders Data (Same query we saw in the earlier demonstration)

1. Select the **Data** tab in your dashboard.

2. Select **Add SQL dataset**.

3. In the query window, use the dropdowns to set your default catalog and schema:

   - Select your catalog **labuser_UNIQUE_ID**

   - Select schema **aibi_dashboards**




4. Add the following query as the source for your dashboard.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <div class="code-block" data-language="sql">
SELECT
  c.customername,
  c.province,
  o.orderdate,
  YEAR(o.orderdate) AS year,
  p.productname,
  o.orderamt,
  o.quantity,
  o.salesrep
FROM ca_orders AS o
  INNER JOIN ca_customers AS c
  ON o.customerid = c.customerid
  INNER JOIN ca_products AS p
  ON o.productid = p.productid;
  </div>
</div>

5. Click **Run** to confirm the query executes. Then examine the results.

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size: 14pt;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:16x;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

6. In the **Data** tab, rename the dataset to `FullOrdersData`.

    - **NOTE:** Click the **kebab menu** next to the dataset and choose **Rename**.


#### Checkpoint

| customername            | province | orderdate  | year | productname | orderamt | quantity | salesrep |
|-------------------------|----------|------------|------|-------------|----------|----------|----------|
| Nova Motor Vehicles     | AB       | 2021-11-16 | 2021 | Grinder     | 61715.5  | 350      | Hannah   |
| Terrestrial Consulting  | ON       | 2021-11-16 | 2021 | Planer      | 14606.06 | 323      | Sam      |
| Zenith Health           | MB       | 2021-11-16 | 2021 | Mill        | 3651.03  | 359      | Ingrid   |
| ...                     | ...      | ...        | ...  | ...         | ...      | ...      | ...      |

### E3. Add a Dataset for Target vs. Actual Sales (Same query we saw in the earlier demonstration)

Combine **ca_sales_targets** with **ca_orders** so the dashboard can show:
  - Each year's target next to the actual revenue
  - The variance between target and actual
  - If we HIT / MISS the target.

1. Still in the **Data** tab, select **Add SQL dataset**.

2. Select your default catalog and schema:
    - **Catalog**: **labuser_UNIQUE_ID**
    - **Schema**: **aibi_dashboards**

3. Enter the following query into the query editor:
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <div class="code-block" data-language="sql">
SELECT
  t.target_set_date,
  t.target_year,
  t.sales_target,
  ROUND(SUM(o.orderamt), 2)                                       AS actual_sales,
  ROUND(SUM(o.orderamt) - t.sales_target, 2)                      AS variance,
  ROUND((SUM(o.orderamt) - t.sales_target) / t.sales_target * 100, 2) AS variance_pct,
  CASE
    WHEN SUM(o.orderamt) >= t.sales_target THEN 'HIT'
    ELSE 'MISS'
  END                                                             AS status,
  t.notes
FROM ca_sales_targets AS t
  INNER JOIN ca_orders AS o
  ON YEAR(o.orderdate) = t.target_year
GROUP BY t.target_set_date, t.target_year, t.sales_target, t.notes
ORDER BY t.target_year DESC;
  </div>
</div>

4. Click **Run** to execute the query and examine the results.

5. Rename the **Dataset** to `SalesTargets`.

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size: 14pt;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:16x;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

#### Checkpoint

| target_set_date | target_year | sales_target | actual_sales | variance     | variance_pct | status | notes                                                                 |
|-----------------|-------------|--------------|--------------|--------------|--------------|--------|-----------------------------------------------------------------------|
| 2024-01-10      | 2024        | 86000000     | 67827541.42  | -18172458.58 | -21.13       | MISS   | Conservative hold-the-line plan after soft Q4 2023.                  |
| 2023-01-08      | 2023        | 87000000     | 82928941.68  | -4071058.32  | -4.68        | MISS   | Expected 8% YoY growth over 2022 actuals.                            |
| 2022-01-05      | 2022        | 80000000     | 84114356.07  | 4114356.07   | 5.14         | HIT    | First full operating year; aggressive growth plan.                   |
| 2021-11-01      | 2021        | 9000000      | 10771691.88  | 1771691.88   | 19.69        | HIT    | Partial-year target (Q4 only; business launched mid-November).       |

## Conclusion

In this demo, you:

1. Created and named a new AI/BI Dashboard (`FirstName-LastInitials - Canadian Sales`).
2. Selected a **Serverless SQL Warehouse** as the dashboard's compute.
3. Added a **pass-through** dataset (**Opportunities**) that pulls `ca_opportunities` directly as-is.
4. Added a **multi-table join** dataset (**FullOrdersData**) combining `ca_orders`, `ca_customers`, and `ca_products`.
5. Added an **aggregation** dataset (**SalesTargets**) that joins `ca_sales_targets` with `ca_orders`, including variance and a HIT / MISS flag per year.
6. Named all three datasets so the dashboard is organized and easy to reuse.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>